# NB05_DQ — Đánh giá chất lượng dữ liệu (mục 3.6)
*Phụ trách: Huy*

**Cách chạy:** mở trên Colab → Runtime → Run all. Notebook tự tải 6 tệp theo ID từ thư mục Drive của nhóm (`staging/`, `NB02_bom_final/`).

Đánh giá theo 5 tiêu chí DAMA UK (2013): Đầy đủ, Hợp lệ, Duy nhất, Nhất quán, Chính xác. Ô cuối in Bảng kết quả dùng cho Bảng 3.i trong báo cáo.

In [ ]:
# ==== NB05_DQ — Đánh giá chất lượng dữ liệu (mục 3.6) ====
# Chạy trên Colab: Runtime → Run all. Tải 6 tệp theo ID từ thư mục Drive của nhóm.
# Chạy ngoài Colab: đặt biến môi trường NB05_BASE trỏ tới thư mục có sẵn các tệp CSV.
import os
import numpy as np
import pandas as pd

FILE_IDS = {
    'sales_clean.csv':      '1XZYQ7rg0pblDnwYuR505OHlrfd0lXSBH',   # staging/
    'products_drinks.csv':  '1RLQdVnBkWOdeXrDLejEVwztAhY8JQcGf',   # staging/
    'ingredient.csv':       '1eW-EZo2Su3G6iZCG-5h-NDB2wHoLlYSj',   # NB02_bom_final/
    'bom.csv':              '1tHTD6iWiOxTBayWvqg3z0X3ti9KysWOo',   # NB02_bom_final/
    'inventory_weekly.csv': '1Im_KEKdN-ZeSCSWYW00_i6rQfLYGeX8V',   # staging/
    'waste_log.csv':        '1xoaMWuvzEKOQTkJ8XR0dXRArioYTX-dz',   # staging/
}
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
BASE = '/content/dq_input' if IN_COLAB else os.environ.get('NB05_BASE', '.')
os.makedirs(BASE, exist_ok=True)
for name, fid in FILE_IDS.items():
    p = os.path.join(BASE, name)
    if not os.path.exists(p) and IN_COLAB:
        import gdown
        gdown.download(id=fid, output=p, quiet=True)

def load(name, **kw):
    p = os.path.join(BASE, name)
    return pd.read_csv(p, encoding='utf-8-sig', **kw) if os.path.exists(p) else None

sales = load('sales_clean.csv', dtype={'transaction_time': str})
pdk = load('products_drinks.csv')
ing = load('ingredient.csv')
bom = load('bom.csv')
inv = load('inventory_weekly.csv')
waste = load('waste_log.csv')
for d in (ing, bom, inv, waste):
    d['ingredient_id'] = d['ingredient_id'].astype(str).str.strip()

TOL = 1e-3
rows = []
def add(criterion, test, files, result, ok, note_if_fail='Không đạt'):
    rows.append({'Tiêu chí': criterion, 'Phép kiểm': test, 'Tệp': files,
                 'Kết quả': result, 'Đánh giá': 'Đạt' if ok else note_if_fail})

fmt = lambda n: f'{int(n):,}'.replace(',', '.')


In [ ]:
# ---------- 1. TÍNH ĐẦY ĐỦ ----------
if sales is not None:
    raw_cols = ['transaction_id', 'transaction_date', 'transaction_time', 'transaction_qty', 'store_id',
                'store_location', 'product_id', 'unit_price', 'product_category', 'product_type', 'product_detail']
    n_null = int(sales[raw_cols].isna().sum().sum())
    add('Đầy đủ', 'Ô trống trong 11 cột gốc', 'sales_clean', f'{n_null}/{fmt(len(sales))} dòng có ô trống', n_null == 0)
others = {'products_drinks': pdk, 'ingredient': ing, 'bom': bom, 'inventory_weekly': inv, 'waste_log': waste}
n_null_o = sum(int(d.isna().sum().sum()) for d in others.values())
add('Đầy đủ', 'Ô trống trong 5 tệp còn lại', ', '.join(others), f'{n_null_o} ô trống', n_null_o == 0)
n_bom = bom['product_id'].nunique()
add('Đầy đủ', 'Mỗi đồ uống có định mức BOM', 'bom', f'{n_bom}/{len(pdk)} món; {len(bom)} dòng',
    set(bom['product_id']) == set(pdk['product_id']))
pairs = inv.groupby(['store_id', 'ingredient_id'])['week_no'].nunique()
n_weeks = inv['week_no'].nunique()
add('Đầy đủ', 'Mỗi cặp cửa hàng × NVL đủ số tuần kho', 'inventory_weekly',
    f'{len(pairs)} cặp × {n_weeks} tuần = {fmt(len(inv))} dòng', (pairs == n_weeks).all() and len(inv) == len(pairs) * n_weeks)


In [ ]:
# ---------- 2. TÍNH HỢP LỆ ----------
SHIFTS = {'Sáng', 'Chiều', 'Tối'}
if sales is not None:
    bad = int(((sales['transaction_qty'] <= 0) | (sales['unit_price'] <= 0)).sum())
    add('Hợp lệ', 'transaction_qty > 0 và unit_price > 0', 'sales_clean', f'{bad} dòng vi phạm', bad == 0)
    d = pd.to_datetime(sales['transaction_date'])
    n_days = d.dt.normalize().nunique()
    ok = d.min() >= pd.Timestamp('2023-01-01') and d.max() <= pd.Timestamp('2023-06-30')
    add('Hợp lệ', 'Ngày và giờ nằm trong kỳ quan sát', 'sales_clean',
        f'{d.min():%d/%m}–{d.max():%d/%m/%Y}; {sales.transaction_time.min()}–{sales.transaction_time.max()}; {n_days} ngày', ok)
    bad_shift_s = int((~sales['shift'].isin(SHIFTS)).sum())
else:
    bad_shift_s = 0
bad_shift_w = int((~waste['shift'].isin(SHIFTS)).sum())
add('Hợp lệ', 'Mọi dòng thuộc một trong 3 ca', 'sales_clean, waste_log',
    f'{bad_shift_s + bad_shift_w} dòng ngoài 3 ca', bad_shift_s + bad_shift_w == 0)
ok = (ing['unit'].isin(['g', 'ml', 'cai']).all() and (ing['unit_cost_usd'] > 0).all()
      and ing['yield_pct'].between(0.01, 100).all() and (bom['quantity_ep'] > 0).all()
      and (bom['quantity_basis'] == 'EP').all())
add('Hợp lệ', 'Miền giá trị của NVL và định mức', 'ingredient, bom',
    f"Đơn vị ∈ {{g, ml, cai}}; đơn giá > 0; yield {ing.yield_pct.min():g}–{ing.yield_pct.max():g}%; "
    f"quantity_ep > 0, cùng cơ sở EP", ok)
qty_cols = ['opening_qty', 'receipt_qty', 'actual_usage', 'closing_qty', 'theoretical_usage', 'recorded_waste']
n_neg = int((inv[qty_cols] < 0).any(axis=1).sum())
add('Hợp lệ', 'Lượng tồn, nhập, tiêu hao, hủy không âm', 'inventory_weekly',
    f'{n_neg}/{fmt(len(inv))} dòng có giá trị âm', n_neg == 0)
wd = pd.to_datetime(waste['waste_date'])
bad = int((waste['waste_qty'] <= 0).sum() + (~wd.between('2023-01-01', '2023-06-30')).sum())
add('Hợp lệ', 'waste_qty > 0 và ngày hủy trong kỳ', 'waste_log',
    f'{bad}/{fmt(len(waste))} dòng vi phạm; {wd.min():%d/%m}–{wd.max():%d/%m/%Y}', bad == 0)


In [ ]:
# ---------- 3. TÍNH DUY NHẤT ----------
if sales is not None:
    n_dup_id = int(sales['transaction_id'].duplicated().sum())
    add('Duy nhất', 'transaction_id không trùng', 'sales_clean', f'{n_dup_id} mã trùng', n_dup_id == 0)
    other_cols = [c for c in raw_cols if c != 'transaction_id']
    n_dup_row = int(sales.duplicated(subset=other_cols).sum())
    add('Duy nhất', 'Dòng trùng mọi trường trừ transaction_id', 'sales_clean',
        f'{n_dup_row} dòng ({n_dup_row / len(sales) * 100:.2f}%)'.replace('.', ','), n_dup_row == 0, 'Cần lưu ý')
keys = [(pdk, ['product_id']), (ing, ['ingredient_id']), (bom, ['product_id', 'ingredient_id']),
        (inv, ['store_id', 'ingredient_id', 'week_start']), (waste, ['waste_id'])]
n_dup_k = sum(int(d.duplicated(k).sum()) for d, k in keys)
add('Duy nhất', 'Khóa của 5 tệp còn lại', 'products_drinks, ingredient, bom, inventory_weekly, waste_log',
    'product_id (41), ingredient_id (25), (product_id, ingredient_id) (178), '
    '(store_id, ingredient_id, week_start) (1.755), waste_id (5.042): ' + f'{n_dup_k} trùng', n_dup_k == 0)


In [ ]:
# ---------- 4. TÍNH NHẤT QUÁN ----------
ids = set(ing['ingredient_id'])
orphan = len((set(bom.ingredient_id) | set(inv.ingredient_id) | set(waste.ingredient_id)) - ids)
orphan += len(set(bom.product_id) ^ set(pdk.product_id))
add('Nhất quán', 'Toàn vẹn tham chiếu mã NVL và mã món', 'bom, inventory_weekly, waste_log, ingredient, products_drinks',
    f'{orphan} mã mồ côi; {n_bom}/{len(pdk)} món trong BOM khớp products_drinks', orphan == 0)
bal = (inv.opening_qty + inv.receipt_qty - inv.actual_usage - inv.closing_qty).abs().max()
add('Nhất quán', 'Tồn đầu + nhập − tiêu hao thực tế = tồn cuối', 'inventory_weekly',
    f'Sai lệch lớn nhất {bal:.1e}', bal <= TOL)
s = inv.sort_values(['store_id', 'ingredient_id', 'week_no'])
gap = (s.groupby(['store_id', 'ingredient_id'])['opening_qty'].shift(-1) - s['closing_qty']).abs()
n_gap = int((gap > TOL).sum())
add('Nhất quán', 'Tồn cuối tuần t = tồn đầu tuần t + 1', 'inventory_weekly', f'{n_gap} dòng lệch', n_gap == 0)
k = ['store_id', 'ingredient_id', 'week_start']
j = inv.set_index(k)['recorded_waste'].to_frame().join(waste.groupby(k)['waste_qty'].sum(), how='outer').fillna(0)
d_w = (j['recorded_waste'] - j['waste_qty']).abs().max()
add('Nhất quán', 'Tổng phiếu hủy theo cửa hàng × NVL × tuần = recorded_waste', 'waste_log, inventory_weekly',
    f'Sai lệch lớn nhất {d_w:.1e}', d_w <= TOL)
cat = ing[['ingredient_id', 'ingredient_name', 'unit', 'unit_cost_usd']]
n_mis = 0
for d in (inv, waste):
    m = d.merge(cat, on='ingredient_id', suffixes=('', '_cat'))
    n_mis += int(((m.ingredient_name != m.ingredient_name_cat) | (m.unit != m.unit_cat)
                  | ((m.unit_cost_usd - m.unit_cost_usd_cat).abs() > 1e-9)).sum())
add('Nhất quán', 'Tên, đơn vị, đơn giá NVL giống danh mục', 'inventory_weekly, waste_log', f'{n_mis} dòng lệch', n_mis == 0)
t = pd.to_datetime(waste['waste_datetime'])
shift_calc = np.select([t.dt.hour <= 10, t.dt.hour <= 15], ['Sáng', 'Chiều'], 'Tối')
week_calc = (t.dt.normalize() - pd.to_timedelta(t.dt.dayofweek, unit='D')).dt.strftime('%Y-%m-%d')
n_bad = int(((shift_calc != waste['shift']) | (week_calc != waste['week_start'])).sum())
add('Nhất quán', 'Ca và tuần của phiếu hủy khớp thời điểm hủy', 'waste_log', f'{n_bad}/{fmt(len(waste))} dòng lệch', n_bad == 0)
b = bom.merge(ing[['ingredient_id', 'yield_pct']], on='ingredient_id')
b['quantity_ap'] = b['quantity_ep'] / (b['yield_pct'] / 100)
b = b.merge(pdk[['product_id', 'qty_sold']], on='product_id')
expected = (b['quantity_ap'] * b['qty_sold']).groupby(b['ingredient_id']).sum()
actual = inv.groupby('ingredient_id')['theoretical_usage'].sum()
cmp_ = pd.concat([expected, actual], axis=1, keys=['exp', 'act']).dropna()
max_pct = ((cmp_.act - cmp_.exp).abs() / cmp_.exp * 100).max()
add('Nhất quán', 'Tiêu hao lý thuyết = Σ(số ly bán × định mức AP) theo NVL',
    'inventory_weekly, bom, ingredient, products_drinks',
    f'{len(cmp_)}/{inv.ingredient_id.nunique()} NVL khớp; lệch lớn nhất {max_pct:.5f}%'.replace('.', ','), max_pct < 0.001)


In [ ]:
# ---------- 5. TÍNH CHÍNH XÁC ----------
if sales is not None:
    ok = len(sales) == 149_116 and np.isclose(sales['revenue'].sum(), 698_812.33, atol=0.005)
    add('Chính xác', 'Số dòng và tổng doanh thu khớp nguồn', 'sales_clean',
        f"{fmt(len(sales))} dòng; {sales['revenue'].sum():,.2f} USD".replace(',', 'X').replace('.', ',').replace('X', '.'), ok)
ok = (pdk['n_lines'].sum() == 115_333 and pdk['qty_sold'].sum() == 176_444
      and np.isclose(pdk['revenue'].sum(), 538_774.40, atol=0.005) and np.isclose(pdk['menu_mix_pct'].sum(), 100, atol=1e-4))
add('Chính xác', 'Số liệu tổng hợp 41 đồ uống khớp dữ liệu dòng', 'products_drinks',
    '115.333 dòng; 176.444 ly; 538.774,40 USD; tổng MM% = 100%', ok)
d1 = (inv.actual_usage * inv.unit_cost_usd - inv.actual_cogs_usd).abs().max()
d2 = (waste.waste_qty * waste.unit_cost_usd - waste.waste_value_usd).abs().max()
add('Chính xác', 'Tính lại các cột giá trị tiền', 'inventory_weekly, waste_log',
    f'actual_cogs_usd, waste_value_usd lệch ≤ {max(d1, d2):.5f} USD (làm tròn 4 chữ số)'.replace('.', ','), max(d1, d2) < TOL)

result = pd.DataFrame(rows)
result.index = range(1, len(result) + 1)
pd.set_option('display.max_colwidth', 120)
print(result.to_string())
print('\nTổng:', len(result), 'phép kiểm |', result['Đánh giá'].value_counts().to_dict())
